# 依存構造解析の精度を UAS と LAS で評価する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_parsing/dep_eval.ipynb)

NLP クックブック 14-14 ／ 構文解析

依存構造パーザの出力を正解のツリーバンクと比べて、係り先の正解率 (UAS) と、ラベルまで含めた正解率 (LAS) を計算したい。

In [1]:
!pip install -q stanza conllu

In [2]:
import urllib.request
import stanza
from conllu import parse

url = "https://raw.githubusercontent.com/UniversalDependencies/UD_Japanese-GSD/master/ja_gsd-ud-test.conllu"
gold = parse(urllib.request.urlopen(url).read().decode("utf-8"))[:150]     # UD Japanese-GSD のテストデータの先頭 150 文
gold = [[t for t in sent if isinstance(t["id"], int)] for sent in gold]    # 複数語トークンなどの行を除く
print(len(gold), "文", sum(map(len, gold)), "語")

# 評価では単語分割を正解に揃える (分割が違うと、係り受けを 1 対 1 で比べられない)
stanza.download("ja", verbose=False)
nlp = stanza.Pipeline("ja", processors="tokenize,pos,lemma,depparse", tokenize_pretokenized=True, verbose=False)
pred = nlp([[t["form"] for t in sent] for sent in gold])

150 文 3226 語


In [3]:
from collections import Counter

total = uas = las = 0
by_rel, by_rel_ok = Counter(), Counter()
errors = Counter()
for g_sent, p_sent in zip(gold, pred.sentences):
    for g, p in zip(g_sent, p_sent.words):
        if g["upos"] == "PUNCT":           # 句読点を除いて評価する流儀
            continue
        total += 1
        head_ok = g["head"] == p.head
        uas += head_ok
        las += head_ok and g["deprel"] == p.deprel
        by_rel[g["deprel"]] += 1
        by_rel_ok[g["deprel"]] += head_ok and g["deprel"] == p.deprel
        if head_ok and g["deprel"] != p.deprel:
            errors[g["deprel"], p.deprel] += 1
print(f"UAS = {uas / total:.4f}  LAS = {las / total:.4f}  ({total} 語)")
print("関係ごとの LAS:")
for rel, n in by_rel.most_common(12):
    print(f"  {rel:12} {by_rel_ok[rel] / n:.3f} ({n})")
print("係り先は合っているがラベルが違う:", errors.most_common(5))

UAS = 0.9363  LAS = 0.9250  (2840 語)
関係ごとの LAS:
  case         0.981 (689)
  compound     0.956 (318)
  aux          0.965 (284)
  obl          0.839 (217)
  mark         0.965 (171)
  nmod         0.896 (164)
  fixed        0.980 (150)
  root         0.973 (150)
  nsubj        0.835 (133)
  acl          0.891 (128)
  advcl        0.711 (121)
  obj          0.978 (90)
係り先は合っているがラベルが違う: [(('advcl', 'acl'), 3), (('advmod', 'obl'), 2), (('nsubj:outer', 'nsubj'), 2), (('advcl', 'obl'), 2), (('cop', 'case'), 2)]


In [4]:
# 文単位の完全一致率と、文の長さごとの精度
exact, buckets = 0, {}
for g_sent, p_sent in zip(gold, pred.sentences):
    ok = [g["head"] == p.head for g, p in zip(g_sent, p_sent.words) if g["upos"] != "PUNCT"]
    exact += all(ok)
    key = "〜10 語" if len(ok) <= 10 else "11〜30 語" if len(ok) <= 30 else "31 語〜"
    buckets.setdefault(key, []).extend(ok)
print(f"文単位の完全一致率: {exact / len(gold):.3f}")
print({k: round(sum(v) / len(v), 3) for k, v in sorted(buckets.items())})

文単位の完全一致率: 0.493
{'11〜30 語': 0.943, '31 語〜': 0.922, '〜10 語': 0.96}


- UAS (Unlabeled Attachment Score) は係り先が合っている語の割合、LAS (Labeled Attachment Score) は係り先とラベルの両方が合っている語の割合。
- 句読点を含めるかどうかで値が変わる。CoNLL の公式評価は句読点も含めるので、比較するときは流儀を揃える。
- 生のテキストから解析した場合は、単語分割の誤りも含めて評価する必要がある。その場合は、文字位置で単語を対応付ける CoNLL 2018 Shared Task の公式スクリプト (`conll18_ud_eval.py`) を使う。
- 日本語では `case` (助詞) や `aux` (助動詞) のように、ほぼ機械的に決まる関係が全体の精度を押し上げる。内容語の係り受け (`nsubj`、`obl`、`acl` など) を個別に見るのが大事。